In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
from pathlib import Path

RAW_DIR = Path("/kaggle/working/nixiebomb_raw")

files = list(RAW_DIR.glob("*.txt"))

print("Text files:", len(files))
print("Total size (MB):", round(
    sum(f.stat().st_size for f in files) / (1024 * 1024), 2
))
print("Last file:", files[-1].name if files else "None")

In [ ]:
from pathlib import Path

RAW_DIR = Path("/kaggle/working/nixiebomb_raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

count = 0
file_index = 0
examples_per_file = 1000

for example in ds:
    comment = example.get("comment", "")
    code = example.get("code", "")

    if not comment.strip() or not code.strip():
        continue

    if count % examples_per_file == 0:
        output_file = RAW_DIR / f"codes_{file_index:04d}.txt"
        file_index += 1

    with output_file.open("a", encoding="utf-8") as f:
        f.write(comment + "\n\n" + code + "\n\n")

    count += 1

    if count % 10000 == 0:
        print(f"Saved {count:,} examples")

print(f"\nFinished: {count:,} examples")
print(f"Files created: {file_index}")
print(f"Location: {RAW_DIR}")

In [ ]:
!pip install -q datasets

from datasets import load_dataset

ds = load_dataset(
    "sentence-transformers/codesearchnet",
    "pair",
    split="train",
    streaming=True
)

print("Dataset connected!")
print("First example:")
print(next(iter(ds)))

In [ ]:
!git clone --depth 1 --branch Brain https://github.com/Salt-the-Fries/NIXIEBOMB.git

In [ ]:
%cd /kaggle/working/NIXIEBOMB
!ls

In [ ]:
!pip install -q "tokenizers>=0.20" "numpy>=1.26"

In [ ]:
import sys
import subprocess

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "tokenizers>=0.20", "numpy>=1.26"],
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print("Installation output:")
print(result.stdout)
print("Errors:")
print(result.stderr)

In [ ]:
import torch
from core.config import ModelConfig
from core.model import NixieBombCore

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

# Create a tiny network with RANDOM initialization
config = ModelConfig.tiny()
model = NixieBombCore(config)

# Give it a small batch of random token IDs
tokens = torch.randint(0, config.vocab_size, (1, 16))

# Run the network
with torch.no_grad():
    result = model(tokens)

print("Output shape:", result["logits"].shape)
print("Parameters:", f"{model.parameter_count():,}")
print("All values finite:", torch.isfinite(result["logits"]).all().item())
print("NIXIEBOMB neural network test complete!")

In [ ]:
import os

print("=== KAGGLE DATASETS ===")

for folder, dirs, files in os.walk("/kaggle/input"):
    if files:
        print("\nFolder:", folder)
        print("Files:", len(files))
        print("Examples:", files[:5])

print("\n=== NIXIEBOMB SOURCE FILES ===")

for folder, dirs, files in os.walk("/kaggle/working/NIXIEBOMB/core"):
    for file in files:
        if file.endswith((".py", ".md")):
            print(os.path.join(folder, file))

In [ ]:
from pathlib import Path

RAW_DIR = Path("/kaggle/working/nixiebomb_raw")
files = sorted(RAW_DIR.glob("*.txt"))

print("Files:", len(files))
print("First 5:", [f.name for f in files[:5]])
print("Last 5:", [f.name for f in files[-5:]])

for f in files[-3:]:
    print(f"\n{f.name}: {f.stat().st_size / 1024:.1f} KB")
    print(f.read_text(encoding="utf-8")[:250])

In [ ]:
!python -m core.prepare_data --help

In [ ]:
%cd /kaggle/working/NIXIEBOMB

!python -m core.prepare_data \
    --input /kaggle/working/nixiebomb_raw \
    --out /kaggle/working/nixiebomb_processed \
    --vocab-size 16384 \
    --val-fraction 0.05

In [ ]:
from pathlib import Path

paths = {
    "Original dataset": Path("/kaggle/working/nixiebomb_raw"),
    "Prepared dataset": Path("/kaggle/working/nixiebomb_processed"),
}

for name, path in paths.items():
    print(f"\n{name}:")
    print("Exists:", path.exists())

    if path.exists():
        for filename in ["train.bin", "val.bin", "tokenizer.json", "manifest.json"]:
            f = path / filename
            if f.exists():
                print(f"  {filename}: {f.stat().st_size / (1024**2):.2f} MB")

In [16]:
import torch
from pathlib import Path

data = Path("/kaggle/working/nixiebomb_processed")

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GiB")

for name in ["train.bin", "val.bin", "tokenizer.json"]:
    p = data / name
    print(name, "EXISTS" if p.exists() else "MISSING",
          f"{p.stat().st_size / 1024**2:.2f} MB" if p.exists() else "")

CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GiB
train.bin EXISTS 379.95 MB
val.bin EXISTS 19.80 MB
tokenizer.json EXISTS 1.06 MB


In [17]:
%cd /kaggle/working/NIXIEBOMB

!python -m core.train \
  --train /kaggle/working/nixiebomb_processed/train.bin \
  --val /kaggle/working/nixiebomb_processed/val.bin \
  --out /kaggle/working/nixiebomb_core_pilot \
  --steps 20 \
  --seq-len 256 \
  --batch-size 1 \
  --grad-accum 4 \
  --warmup 5 \
  --eval-every 5 \
  --eval-batches 2 \
  --save-every 5 \
  --observe-every 5 \
  --precision auto

/kaggle/working/NIXIEBOMB
step=0 train_loss=9.8510 val_loss=9.7679 lr=6.000e-05 precision=bf16
observatory=/kaggle/working/nixiebomb_core_pilot/observatory/latest.json
saved=/kaggle/working/nixiebomb_core_pilot/latest.pt
step=5 train_loss=8.0853 val_loss=8.0759 lr=3.000e-04 precision=bf16
observatory=/kaggle/working/nixiebomb_core_pilot/observatory/latest.json
saved=/kaggle/working/nixiebomb_core_pilot/latest.pt
step=10 train_loss=7.8633 val_loss=7.3280 lr=2.250e-04 precision=bf16
observatory=/kaggle/working/nixiebomb_core_pilot/observatory/latest.json
saved=/kaggle/working/nixiebomb_core_pilot/latest.pt
step=15 train_loss=7.4169 val_loss=7.2782 lr=7.500e-05 precision=bf16
observatory=/kaggle/working/nixiebomb_core_pilot/observatory/latest.json
saved=/kaggle/working/nixiebomb_core_pilot/latest.pt
saved=/kaggle/working/nixiebomb_core_pilot/latest.pt


In [18]:
from pathlib import Path
import torch, json

out = Path("/kaggle/working/nixiebomb_core_pilot")

checkpoint = torch.load(
    out / "latest.pt",
    map_location="cpu",
    weights_only=False
)

print("Checkpoint exists:", (out / "latest.pt").exists())
print("Checkpoint size:", round(
    (out / "latest.pt").stat().st_size / 1024**2, 2
), "MB")
print("Completed step:", checkpoint["step"])
print("Model vocabulary:", checkpoint["config"]["vocab_size"])
print("Precision:", checkpoint["precision"])

report = json.loads(
    (out / "observatory/latest.json").read_text()
)

print("Observatory step:", report["step"])
print("Parameter count:", report["parameters"])

Checkpoint exists: True
Checkpoint size: 1008.34 MB
Completed step: 19
Model vocabulary: 16384
Precision: bf16
Observatory step: 15
Parameter count: 88099584
